## Import Dataset

In [1]:
import pandas as pd
df = pd.read_csv("../../raw_data/recipes_ingredients.csv")

## Checking stuff

In [2]:
df.shape

(500471, 9)

In [3]:
df.head()

,id,name,description,ingredients,ingredients_raw,steps,servings,serving_size,tags
0,71247,Cherry Streusel Cobbler,"I haven't made this in years, so I'm just gues...","[""cherry pie filling"", ""condensed milk"", ""melt...","[""2 (21 ounce) cans cherry pie filling"",""2...","[""Preheat oven to 375°F."", ""Spread cherry pie ...",6.0,1 (347 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
1,76133,Reuben and Swiss Casserole Bake,I think this is even better than a reuben sand...,"[""corned beef chopped"", ""sauerkraut cold water...","[""1/2-1 lb corned beef, cooked and choppe...","[""Set oven to 350 degrees F."", ""Butter a 9 x 1...",4.0,1 (207 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
2,503816,Yam-Pecan Recipe,A lady I work with heard me taking about ZWT a...,"[""unsalted butter"", ""vegetable oil"", ""all - pu...","[""3/4 cup unsalted butter, at room tempera...","[""Preheat oven to 350°F In a mixing bowl, usi...",8.0,1 (198 g),"[""time-to-make"", ""course"", ""main-ingredient"", ..."
3,418749,Tropical Orange Layer Cake,An easy and delicious cake. Great for a summ...,"[""orange cake mix"", ""instant vanilla pudding"",...","[""1 (18 ounce) pkge.orange cake mix"",""1 (3...","[""In a large mixing bowl, combine the first 6 ...",16.0,1 (191 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
4,392934,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,I was searching the web for something like thi...,"[""butter"", ""brown sugar"", ""granulated sugar"", ...","[""1/2 cup butter, room temperature "",""1/2 ...","[""Cream butter and sugars together."", ""Blend i...",24.0,1 (26 g),"[""15-minutes-or-less"", ""time-to-make"", ""course..."


## Preprocessing

In [4]:
data = df.dropna()  #enlève les NaN

In [5]:
data = data.drop(columns=["description","id"])  # on enlève les colonnes innutiles
data.head()

,name,ingredients,ingredients_raw,steps,servings,serving_size,tags
0,Cherry Streusel Cobbler,"[""cherry pie filling"", ""condensed milk"", ""melt...","[""2 (21 ounce) cans cherry pie filling"",""2...","[""Preheat oven to 375°F."", ""Spread cherry pie ...",6.0,1 (347 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
1,Reuben and Swiss Casserole Bake,"[""corned beef chopped"", ""sauerkraut cold water...","[""1/2-1 lb corned beef, cooked and choppe...","[""Set oven to 350 degrees F."", ""Butter a 9 x 1...",4.0,1 (207 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
2,Yam-Pecan Recipe,"[""unsalted butter"", ""vegetable oil"", ""all - pu...","[""3/4 cup unsalted butter, at room tempera...","[""Preheat oven to 350°F In a mixing bowl, usi...",8.0,1 (198 g),"[""time-to-make"", ""course"", ""main-ingredient"", ..."
3,Tropical Orange Layer Cake,"[""orange cake mix"", ""instant vanilla pudding"",...","[""1 (18 ounce) pkge.orange cake mix"",""1 (3...","[""In a large mixing bowl, combine the first 6 ...",16.0,1 (191 g),"[""60-minutes-or-less"", ""time-to-make"", ""course..."
4,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,"[""butter"", ""brown sugar"", ""granulated sugar"", ...","[""1/2 cup butter, room temperature "",""1/2 ...","[""Cream butter and sugars together."", ""Blend i...",24.0,1 (26 g),"[""15-minutes-or-less"", ""time-to-make"", ""course..."


## Fonction pour transformer les strings en liste.
## on applique sur les colonnes concernés (ingredients, ingredients_raw, steps, tags)

In [6]:
import ast

errors = []

def safe_literal_eval(value):
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError) as e:
        errors.append((value, str(e)))
        return value

data["ingredients"] = data["ingredients"].apply(safe_literal_eval)
data["ingredients_raw"] = data["ingredients_raw"].apply(safe_literal_eval)
data["steps"] = data["steps"].apply(safe_literal_eval)
data["tags"] = data["tags"].apply(safe_literal_eval)


In [7]:
type(data.iloc[0].steps) #vérification

list

In [8]:
data.shape #shape avant

(497563, 7)

## On enlève les lignes sans ingrédients, ingredients_raw, steps et tags

In [9]:
data = data[data["ingredients"].apply(len) > 0]
data = data[data["ingredients_raw"].apply(len) > 0]
data = data[data["steps"].apply(len) > 0]
data = data[data["tags"].apply(len) > 0]

In [10]:
data.shape

(489757, 7)

## On teste avec une liste d'ingrédient fictive

In [11]:
user_ingredients = ["tomato", "chicken", "onion", "garlic", "apple", "sugar", "cheese", "bread"]
top_recipes = data.copy() # New Dataset pour rank en fonction de la liste

# Fonction pour compter le nombre de matchs entre la liste et le dataset

In [12]:
def count_matches(recipe_ingredients, user_ingredients):
    matches = 0

    for user_ing in user_ingredients:
        for recipe_ing in recipe_ingredients:
            if user_ing.lower() in recipe_ing.lower():
                matches += 1
                break

    return matches

# Ajoute au nouveau dataset , une colonne match count pour compter le nombre de matchs

In [13]:
top_recipes["match_count"] = top_recipes["ingredients"].apply(lambda x: count_matches(x, user_ingredients))

In [14]:
top_recipes.sort_values("match_count", ascending=False, inplace=True)

In [15]:
top_recipes.head()

,name,ingredients,ingredients_raw,steps,servings,serving_size,tags,match_count
234291,BBQ Chicken Packed Pita,[boneless skinless chicken thighs breasts thig...,[1 lb boneless skinless chicken thighs (o...,[Prepare a hot grill outside or preheat a gril...,1.0,1 (4047 g),"[weeknight, 60-minutes-or-less, time-to-make, ...",8
109082,Fajita Pita,"[chicken breast, soy sauce, lime juice, canola...","[1 lb chicken breast, skinless, boneless ...","[""Whisk together the marinade ingredients in a...",3.0,1 (296 g),"[time-to-make, main-ingredient, cuisine, prepa...",7
140096,Ultimate Chicken Parmigiana,"[virgin olive oil, virgin olive oil, medium on...","[1/4 cup extra virgin olive oil, plus , 3 ...","[Preheat the oven to 350 degrees F., Coat a sa...",4.0,1 (864 g),"[time-to-make, course, main-ingredient, cuisin...",7
185243,Chicken Parmigiana - Tyler Florence,"[virgin olive oil, medium onion chopped, salt,...","[1/2 cup extra virgin olive oil, divided ,...","[In a large skillet, heat 1/4 c olive oil over...",4.0,1 (809 g),"[celebrity, 60-minutes-or-less, time-to-make, ...",7
391728,Chicken Parmigiana,"[olive oil, brown onion, garlic clove minced, ...","[1 teaspoon olive oil, 1 brown onio...","[Heat oil in a small saucepan on medium, and a...",4.0,1 (539 g),"[time-to-make, course, main-ingredient, prepar...",7


# vérification des ingrédients à la main

In [16]:
liste = top_recipes.iloc[1].ingredients
liste

['chicken breast',
 'soy sauce',
 'lime juice',
 'canola oil',
 'garlic minced',
 'brown sugar',
 'cumin',
 'chili powder',
 'cheddar cheese shredded',
 'onions',
 'salt pepper',
 'tomatoes beefsteak',
 'iceberg lettuce',
 'pita bread']

In [17]:
top_recipes.iloc[0].steps

['Prepare a hot grill outside or preheat a grill pan to high heat and preheat oven to 375 degrees F.',
 'Toss raw chicken in a bowl with rub, a few pieces at a time until all chicken is well coated. Grill chicken on well oiled grill or grill pan for about 2 minutes on each side, until nice grill marks are formed. If chicken pieces are very thick, finish cooking for 5 to 10 minutes more in the preheated oven. Allow chicken to cool a few minutes and then pull apart into nice shreds. Toss chicken with BBQ sauce so that it is well coated but not overly wet. Keep warm in a warm saute pan until ready to use.',
 'Heat olive oil in large saute pan over medium-high heat. Stir in garlic with wooden spoon. After 30 seconds add spinach and turn off heat. Toss spinach in hot oil, when wilted completely season with salt. Set aside.',
 'Spread butter evenly over 1 side of each pita. In a large saute pan, grill each flatbread over medium-high heat, buttered side down (work in batches if necessary) top

# Diviser serving_size en 2 colonnes distinctes

In [18]:
data.head(1)

,name,ingredients,ingredients_raw,steps,servings,serving_size,tags
0,Cherry Streusel Cobbler,"[cherry pie filling, condensed milk, melted ma...","[2 (21 ounce) cans cherry pie filling, 2 ...","[Preheat oven to 375°F., Spread cherry pie fil...",6.0,1 (347 g),"[60-minutes-or-less, time-to-make, course, mai..."


In [19]:
data[["persons", "portion_size"]] = data["serving_size"].str.extract(
    r"(\d+)\s*\(([^)]+)\)")
data.head()

,name,ingredients,ingredients_raw,steps,servings,serving_size,tags,persons,portion_size
0,Cherry Streusel Cobbler,"[cherry pie filling, condensed milk, melted ma...","[2 (21 ounce) cans cherry pie filling, 2 ...","[Preheat oven to 375°F., Spread cherry pie fil...",6.0,1 (347 g),"[60-minutes-or-less, time-to-make, course, mai...",1,347 g
1,Reuben and Swiss Casserole Bake,"[corned beef chopped, sauerkraut cold water, s...","[1/2-1 lb corned beef, cooked and chopped...","[Set oven to 350 degrees F., Butter a 9 x 13-i...",4.0,1 (207 g),"[60-minutes-or-less, time-to-make, course, mai...",1,207 g
2,Yam-Pecan Recipe,"[unsalted butter, vegetable oil, all - purpose...","[3/4 cup unsalted butter, at room temperat...","[Preheat oven to 350°F In a mixing bowl, usin...",8.0,1 (198 g),"[time-to-make, course, main-ingredient, cuisin...",1,198 g
3,Tropical Orange Layer Cake,"[orange cake mix, instant vanilla pudding, ora...","[1 (18 ounce) pkge.orange cake mix, 1 (3 ...","[In a large mixing bowl, combine the first 6 i...",16.0,1 (191 g),"[60-minutes-or-less, time-to-make, course, pre...",1,191 g
4,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,"[butter, brown sugar, granulated sugar, milk, ...","[1/2 cup butter, room temperature , 1/2 c...","[Cream butter and sugars together., Blend in m...",24.0,1 (26 g),"[15-minutes-or-less, time-to-make, course, mai...",1,26 g


In [20]:
data = data.drop(columns="serving_size", axis=1)
data.head()

,name,ingredients,ingredients_raw,steps,servings,tags,persons,portion_size
0,Cherry Streusel Cobbler,"[cherry pie filling, condensed milk, melted ma...","[2 (21 ounce) cans cherry pie filling, 2 ...","[Preheat oven to 375°F., Spread cherry pie fil...",6.0,"[60-minutes-or-less, time-to-make, course, mai...",1,347 g
1,Reuben and Swiss Casserole Bake,"[corned beef chopped, sauerkraut cold water, s...","[1/2-1 lb corned beef, cooked and chopped...","[Set oven to 350 degrees F., Butter a 9 x 13-i...",4.0,"[60-minutes-or-less, time-to-make, course, mai...",1,207 g
2,Yam-Pecan Recipe,"[unsalted butter, vegetable oil, all - purpose...","[3/4 cup unsalted butter, at room temperat...","[Preheat oven to 350°F In a mixing bowl, usin...",8.0,"[time-to-make, course, main-ingredient, cuisin...",1,198 g
3,Tropical Orange Layer Cake,"[orange cake mix, instant vanilla pudding, ora...","[1 (18 ounce) pkge.orange cake mix, 1 (3 ...","[In a large mixing bowl, combine the first 6 i...",16.0,"[60-minutes-or-less, time-to-make, course, pre...",1,191 g
4,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,"[butter, brown sugar, granulated sugar, milk, ...","[1/2 cup butter, room temperature , 1/2 c...","[Cream butter and sugars together., Blend in m...",24.0,"[15-minutes-or-less, time-to-make, course, mai...",1,26 g


## Enlever les recettes "outliers"

In [21]:
data = data[data["servings"] <= 50]
data.head()

,name,ingredients,ingredients_raw,steps,servings,tags,persons,portion_size
0,Cherry Streusel Cobbler,"[cherry pie filling, condensed milk, melted ma...","[2 (21 ounce) cans cherry pie filling, 2 ...","[Preheat oven to 375°F., Spread cherry pie fil...",6.0,"[60-minutes-or-less, time-to-make, course, mai...",1,347 g
1,Reuben and Swiss Casserole Bake,"[corned beef chopped, sauerkraut cold water, s...","[1/2-1 lb corned beef, cooked and chopped...","[Set oven to 350 degrees F., Butter a 9 x 13-i...",4.0,"[60-minutes-or-less, time-to-make, course, mai...",1,207 g
2,Yam-Pecan Recipe,"[unsalted butter, vegetable oil, all - purpose...","[3/4 cup unsalted butter, at room temperat...","[Preheat oven to 350°F In a mixing bowl, usin...",8.0,"[time-to-make, course, main-ingredient, cuisin...",1,198 g
3,Tropical Orange Layer Cake,"[orange cake mix, instant vanilla pudding, ora...","[1 (18 ounce) pkge.orange cake mix, 1 (3 ...","[In a large mixing bowl, combine the first 6 i...",16.0,"[60-minutes-or-less, time-to-make, course, pre...",1,191 g
4,Safe to Eat Raw Chocolate Chip Oreo Cookie &qu...,"[butter, brown sugar, granulated sugar, milk, ...","[1/2 cup butter, room temperature , 1/2 c...","[Cream butter and sugars together., Blend in m...",24.0,"[15-minutes-or-less, time-to-make, course, mai...",1,26 g


## Diminuer la liste des ingrédients et essayer de clean cette colonne

# on compte le nombre "d'ingrédients" différents avant nettoyage

In [22]:
unique_ingredients = (
    data["ingredients"]
    .explode()
    .dropna()
    .unique()
)

len(unique_ingredients)

250291

## on défini des liste de mots récurrents et inutiles

In [23]:
PREP_WORDS = [
    "chopped", "diced", "minced", "sliced",
    "peeled", "crushed", "grated", "shredded",
    "melted", "softened", "divided",
    "finely", "roughly", "thinly",
    "drained", "rinsed", "cooked",
    "uncooked", "boiled", "roasted", "hard",
    "soft"
]

SIZE_WORDS = [
    "small", "medium", "large", "extra-large"
]

QUALITY_WORDS = [
    "fresh", "frozen", "organic",
    "ripe", "optional", "preferred",
    "approximately", "about"
]

UNIT_WORDS = [
    "g", "kg", "gram", "grams",
    "ml", "l", "liter", "liters",
    "cup", "cups",
    "tbsp", "tablespoon", "tablespoons",
    "tsp", "teaspoon", "teaspoons",
    "oz", "ounce", "ounces",
    "lb", "lbs", "pound", "pounds"
]

words_to_remove = (
        PREP_WORDS
        + SIZE_WORDS
        + QUALITY_WORDS
        + UNIT_WORDS
    )

## on créer une fonction pour le .apply , qui va enlever toutes les itérations des mots précédents

In [ ]:
import re
def clean_ingredient(text):
    text = text.lower().strip()


    text = re.sub(r"\b\d+([./]\d+)?\b", " ", text)

    for word in words_to_remove:
        text = re.sub(rf"\b{re.escape(word)}\b", " ", text)


    text = re.sub(r"[,()]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [25]:
data["ingredients_clean"] = data["ingredients"].apply(
    lambda ingredients: [
        clean_ingredient(x)
        for x in ingredients
    ]
)

# on les recompte

In [26]:
unique_clean = (
    data["ingredients_clean"]
    .explode()
    .dropna()
    .unique()
)

len(unique_clean)

211312

# On lemmatize pour enlever les pluriels

In [36]:
import spacy #on construit les versions lemmatized des ingredients uniques

nlp = spacy.load("en_core_web_sm")

lemmatized = []

for doc in nlp.pipe(unique_clean, batch_size=1000):
    lemma = " ".join(token.lemma_ for token in doc)
    lemmatized.append(lemma)

In [ ]:
lemma_mapping = dict(zip(unique_clean, lemmatized))
#on créer un dictionnaire pour le mapping à appliquer sur le Dataframe

In [38]:
data["ingredients_lemmatized"] = data["ingredients_clean"].apply(
    lambda ingredients: [
        lemma_mapping[ingredient]
        for ingredient in ingredients
    ]
)

#et on remplace tout dans le dataframe

In [ ]:
current_ingredients = set(
    data["ingredients_clean"]
    .explode()
    .dropna()
)

mapping_ingredients = set(
    lemma_mapping.keys()
)


0


In [49]:
len(current_ingredients)

211312

In [42]:
print("Raw uniques :", data["ingredients"].explode().nunique())
print("Clean uniques :", data["ingredients_clean"].explode().nunique())
print("Lemma uniques :", data["ingredients_lemmatized"].explode().nunique())

Raw uniques : 250291
Clean uniques : 211312
Lemma uniques : 201787


In [44]:
ingredient_counts = (
    data["ingredients_lemmatized"]
    .explode()
    .value_counts()
)


print("Unique:", len(ingredient_counts))
print("Appearing once:", (ingredient_counts == 1).sum())
print("Appearing <= 2:", (ingredient_counts <= 2).sum())
print("Appearing <= 5:", (ingredient_counts <= 5).sum())
print("Appearing <= 10:", (ingredient_counts <= 10).sum())

Unique: 201787
Appearing once: 151577
Appearing <= 2: 170966
Appearing <= 5: 185527
Appearing <= 10: 191719


In [45]:
cumulative = ingredient_counts.cumsum() / ingredient_counts.sum()

print("80% coverage:", (cumulative <= 0.80).sum())
print("90% coverage:", (cumulative <= 0.90).sum())
print("95% coverage:", (cumulative <= 0.95).sum())

80% coverage: 943
90% coverage: 4800
95% coverage: 30628


In [46]:
most_commun = ingredient_counts.head(4800).index.tolist()

In [47]:
most_commun

['onion',
 'garlic clove',
 'olive oil',
 'vanilla',
 'butter',
 'lemon juice',
 'brown sugar',
 'cinnamon',
 'bake powder',
 'milk',
 'salt pepper',
 'tomato',
 'black pepper',
 'bake soda',
 'parmesan cheese',
 'carrot',
 'water',
 'flour',
 'parsley',
 'vegetable oil',
 'ginger',
 'sour cream',
 'unsalted butter',
 'salt',
 'celery',
 'cumin',
 'cream cheese',
 'green onion',
 'garlic powder',
 'soy sauce',
 'nutmeg',
 'garlic',
 'cheddar cheese',
 'ground beef',
 'cilantro',
 'virgin olive oil',
 'mayonnaise',
 'ground black pepper',
 'worcestershire sauce',
 'mushroom',
 'chicken broth',
 'paprika',
 'granulate sugar',
 'potato',
 'chili powder',
 'lime juice',
 'butter margarine',
 'red onion',
 'all',
 'pecan',
 'powder sugar',
 'cayenne pepper',
 'basil',
 'boneless skinless chicken breast',
 'cornstarch',
 'heavy cream',
 'walnut',
 'red pepper flake',
 'oregano',
 'pepper',
 'dijon mustard',
 'bacon',
 'orange juice',
 'clove garlic',
 'green pepper',
 'egg yolk',
 'red bell 

In [51]:
data.shape

(487306, 10)

## Embedding

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")
most_commun_embeddings = embedder.encode(
    most_commun,
    normalize_embeddings=True,
    show_progress_bar=True
)

/home/mitri/.pyenv/versions/3.10.6/envs/master_shelf/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 150/150 [00:02<00:00, 60.62it/s]


In [53]:
most_commun_embeddings.shape

(4800, 384)

In [55]:
import numpy as np

def closest_ingredients(query, top_k=5):
    query_emb = embedder.encode(
        [query],
        normalize_embeddings=True
    )[0]

    scores = most_commun_embeddings @ query_emb

    best_idx = np.argsort(scores)[::-1][:top_k]

    return [
        (most_commun[i], scores[i])
        for i in best_idx
    ]

In [66]:
queries = [
    "dijon mustard",
    "red onions",
    "chicken breast fillet",
    "cherry tomatoes",
    "dragon fruit"
]

for query in queries:
    print(f"\n{query}:")
    print(closest_ingredients(query))


dijon mustard:
[('dijon mustard', np.float32(0.99999994)), ('mustard dijon', np.float32(0.99081874)), ('dijon style mustard', np.float32(0.96212184)), ('dijon - style mustard', np.float32(0.96070474)), ('country dijon mustard', np.float32(0.92623365))]

red onions:
[('red onion', np.float32(0.9352303)), ('red onion red onion', np.float32(0.92675245)), ('red onion onion', np.float32(0.9165962)), ('red onion white onion', np.float32(0.87098163)), ('red onion ring', np.float32(0.8091462))]

chicken breast fillet:
[('chicken breast fillet', np.float32(1.0000002)), ('chicken fillet', np.float32(0.89703786)), ('chicken thigh fillet', np.float32(0.81749976)), ('chicken chicken breast', np.float32(0.7561429)), ('chicken breast', np.float32(0.74810666))]

cherry tomatoes:
[('cherry tomato', np.float32(0.9440539)), ('tomato cherry tomato', np.float32(0.9116101)), ('cherry tomato tomato', np.float32(0.9111796)), ('grape tomato cherry tomato', np.float32(0.8639805)), ('cherry tomato grape tomato'

In [58]:
import numpy as np

def normalize_with_embedding(ingredient, threshold=0.93):
    # on regarde si il existe déjà dans les most commmuns
    if ingredient in most_commun:
        return ingredient

    # sinon on embed
    ingredient_embedding = embedder.encode(
        [ingredient],
        normalize_embeddings=True
    )[0]

    # On regarde avec le meilleur score
    scores = most_commun_embeddings @ ingredient_embedding

    best_idx = np.argmax(scores)
    best_score = scores[best_idx]
    best_match = most_commun[best_idx]

    # on return le best match
    if best_score >= threshold:
        return best_match

    return ingredient

In [ ]:
unique_to_normalize = (
    data["ingredients_lemmatized"]
    .explode()
    .dropna()
    .unique()
)


In [60]:
normalization_mapping = {
    ingredient: normalize_with_embedding(ingredient)
    for ingredient in unique_to_normalize
}

In [61]:
data["ingredients_normalized"] = data["ingredients_lemmatized"].apply(
    lambda ingredients: [
        normalization_mapping[ingredient]
        for ingredient in ingredients
    ]
)

In [ ]:
import pickle #SAVE LE MAPPING IMPORTANT

with open("../../models/ingredient_mapping.pkl", "wb") as f:
    pickle.dump(normalization_mapping, f)

In [67]:
unique_normalized = (
    data["ingredients_normalized"]
    .explode()
    .dropna()
    .unique()
)
len(unique_normalized)

181003

In [68]:
ingredient_counts_normalized = (
    data["ingredients_normalized"]
    .explode()
    .value_counts()
)

cumulative = (
    ingredient_counts_normalized.cumsum()
    / ingredient_counts_normalized.sum()
)

print("80%:", (cumulative <= 0.80).sum())
print("90%:", (cumulative <= 0.90).sum())
print("95%:", (cumulative <= 0.95).sum())

80%: 896
90%: 3453
95%: 19089


## From scratch !! Important !!


In [ ]:
import ast
import re
import spacy

In [69]:
df_recette = pd.read_csv("../../raw_data/recipes_ingredients.csv")

In [70]:
df_recette = df_recette.drop(columns=["description","id"])  # on enlève les colonnes innutiles
df_recette = df_recette.dropna()

In [ ]:
errors = [] # peut être utilisé pour debug

def safe_literal_eval(value):
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError) as e:
        errors.append((value, str(e)))
        return value

df_recette["ingredients"] = df_recette["ingredients"].apply(safe_literal_eval)
df_recette["ingredients_raw"] = df_recette["ingredients_raw"].apply(safe_literal_eval)
df_recette["steps"] = df_recette["steps"].apply(safe_literal_eval)
df_recette["tags"] = df_recette["tags"].apply(safe_literal_eval)

In [72]:
df_recette = df_recette[df_recette["ingredients"].apply(len) > 0]
df_recette = df_recette[df_recette["ingredients_raw"].apply(len) > 0]
df_recette = df_recette[df_recette["steps"].apply(len) > 0]
df_recette = df_recette[df_recette["tags"].apply(len) > 0]

In [73]:
df_recette[["persons", "portion_size"]] = df_recette["serving_size"].str.extract(
    r"(\d+)\s*\(([^)]+)\)")
df_recette = df_recette.drop(columns="serving_size", axis=1)

In [ ]:
unique_ingredients = (
    df_recette["ingredients"]
    .explode()
    .dropna()
    .unique()
)

251117

In [75]:
PREP_WORDS = [
    "chopped", "diced", "minced", "sliced",
    "peeled", "crushed", "grated", "shredded",
    "melted", "softened", "divided",
    "finely", "roughly", "thinly",
    "drained", "rinsed", "cooked",
    "uncooked", "boiled", "roasted", "hard",
    "soft"
]

SIZE_WORDS = [
    "small", "medium", "large", "extra-large"
]

QUALITY_WORDS = [
    "fresh", "frozen", "organic",
    "ripe", "optional", "preferred",
    "approximately", "about"
]

UNIT_WORDS = [
    "g", "kg", "gram", "grams",
    "ml", "l", "liter", "liters",
    "cup", "cups",
    "tbsp", "tablespoon", "tablespoons",
    "tsp", "teaspoon", "teaspoons",
    "oz", "ounce", "ounces",
    "lb", "lbs", "pound", "pounds"
]

words_to_remove = (
        PREP_WORDS
        + SIZE_WORDS
        + QUALITY_WORDS
        + UNIT_WORDS
    )

In [76]:
def clean_ingredient(text):
    text = text.lower().strip()


    text = re.sub(r"\b\d+([./]\d+)?\b", " ", text)

    for word in words_to_remove:
        text = re.sub(rf"\b{re.escape(word)}\b", " ", text)


    text = re.sub(r"[,()]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [77]:
df_recette["ingredients_clean"] = df_recette["ingredients"].apply(
    lambda ingredients: [
        clean_ingredient(x)
        for x in ingredients
    ]
)

In [ ]:
unique_clean = (
    df_recette["ingredients_clean"]
    .explode()
    .dropna()
    .unique()
)

In [80]:
with open("../../models/ingredient_mapping.pkl", "rb") as f:
    normalization_mapping = pickle.load(f)

In [81]:
df_recette["ingredients_lemmatized"] = df_recette["ingredients_clean"].apply(
    lambda ingredients: [
        lemma_mapping[ingredient]
        for ingredient in ingredients
    ]
)

KeyError: 'long green chili pepperoncini peppers'